# FloodNet NYC – API Ingestion Prototype

Ingests two NYC Open Data (Socrata) datasets and merges them using **DuckDB** (with spatial extension).  
**Polars** is used only for DataFrame manipulations that are more readable than SQL.  
Credentials are loaded from a `.env` file.

| # | Dataset ID | API v3 URL |
|---|------------|------------|
| 1 | `kb2e-tjy3` | `https://data.cityofnewyork.us/api/v3/views/kb2e-tjy3/query.json` |
| 2 | `aq7i-eu5q` | `https://data.cityofnewyork.us/api/v3/views/aq7i-eu5q/query.json` |

## 0 · Setup – env & imports

In [ ]:
# ── Install dependencies (uncomment if needed) ────────────────────────────
# %pip install sodapy duckdb polars python-dotenv requests --quiet

In [ ]:
import os
import json
import requests
import duckdb
import polars as pl
from sodapy import Socrata
from dotenv import load_dotenv

# Load .env from repo root (create one if it doesn't exist – see cell below)
load_dotenv()

# ── Credentials ──────────────────────────────────────────────────────────
SOCRATA_DOMAIN  = os.getenv("SOCRATA_DOMAIN",  "data.cityofnewyork.us")
SOCRATA_APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")   # required for v3 endpoints
SOCRATA_USERNAME  = os.getenv("SOCRATA_USERNAME")     # optional – only for write ops
SOCRATA_PASSWORD  = os.getenv("SOCRATA_PASSWORD")     # optional – only for write ops

# ── Dataset IDs ───────────────────────────────────────────────────────────
DS1_ID = "kb2e-tjy3"
DS2_ID = "aq7i-eu5q"

print(f"Domain : {SOCRATA_DOMAIN}")
print(f"App token set: {bool(SOCRATA_APP_TOKEN)}")

In [ ]:
# ── .env template (run once, then fill in your token) ─────────────────────
env_path = ".env"
if not os.path.exists(env_path):
    template = (
        "SOCRATA_DOMAIN=data.cityofnewyork.us\n"
        "SOCRATA_APP_TOKEN=your_app_token_here\n"
        "# SOCRATA_USERNAME=\n"
        "# SOCRATA_PASSWORD=\n"
    )
    with open(env_path, "w") as f:
        f.write(template)
    print(f"Created {env_path} – fill in your Socrata app token and re-run cell 2.")
else:
    print(f"{env_path} already exists.")

## 1 · Ingest via Socrata API

Two ingestion strategies are provided:
- **1-A** – Standard SODA v2 via `sodapy` (simpler, paginated)
- **1-B** – Socrata v3 query endpoint via `requests` (matches the exact URLs provided)

### 1-A · `sodapy` – SODA v2 (recommended for large datasets)

In [ ]:
def fetch_sodapy(dataset_id: str, limit: int = 50_000) -> list[dict]:
    """
    Fetch all rows from a Socrata dataset using sodapy (SODA v2).
    Paginates automatically if row count > `limit`.
    """
    client = Socrata(
        SOCRATA_DOMAIN,
        SOCRATA_APP_TOKEN,
        username=SOCRATA_USERNAME,
        password=SOCRATA_PASSWORD,
        timeout=60,
    )

    rows, offset = [], 0
    while True:
        page = client.get(dataset_id, limit=limit, offset=offset)
        rows.extend(page)
        if len(page) < limit:
            break
        offset += limit

    client.close()
    print(f"[{dataset_id}] fetched {len(rows):,} rows via SODA v2")
    return rows


raw1_v2 = fetch_sodapy(DS1_ID)
raw2_v2 = fetch_sodapy(DS2_ID)

### 1-B · `requests` – Socrata v3 query endpoint

In [ ]:
V3_BASE = "https://data.cityofnewyork.us/api/v3/views/{dataset_id}/query.json"


def fetch_v3(
    dataset_id: str,
    select: str = "*",
    where: str | None = None,
    order: str | None = None,
    limit: int = 50_000,
) -> list[dict]:
    """
    Fetch rows from the Socrata v3 query endpoint.
    Uses offset pagination to retrieve all records.
    """
    url = V3_BASE.format(dataset_id=dataset_id)
    headers = {"Accept": "application/json"}
    if SOCRATA_APP_TOKEN:
        headers["X-App-Token"] = SOCRATA_APP_TOKEN

    rows, offset = [], 0
    while True:
        params: dict = {"$select": select, "$limit": limit, "$offset": offset}
        if where:
            params["$where"] = where
        if order:
            params["$order"] = order

        resp = requests.get(url, headers=headers, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()

        # v3 response: {"resultSet": {"rows": [...], "columns": [...]}}
        result_set = payload.get("resultSet", payload)
        page = result_set.get("rows", [])
        cols = result_set.get("columns", [])

        # Convert positional rows to dicts if columns are present
        if cols and page and isinstance(page[0], list):
            col_names = [c["name"] for c in cols]
            page = [dict(zip(col_names, r)) for r in page]

        rows.extend(page)
        if len(page) < limit:
            break
        offset += limit

    print(f"[{dataset_id}] fetched {len(rows):,} rows via v3")
    return rows


raw1_v3 = fetch_v3(DS1_ID)
raw2_v3 = fetch_v3(DS2_ID)

## 2 · Load into DuckDB

In [ ]:
# In-memory DuckDB instance (swap ':memory:' for a file path to persist)
con = duckdb.connect(":memory:")

# Install & load spatial extension
con.execute("INSTALL spatial; LOAD spatial;")
print("DuckDB spatial extension loaded.")

In [ ]:
def records_to_duckdb(con: duckdb.DuckDBPyConnection, records: list[dict], table_name: str) -> None:
    """
    Register a list of dicts as a DuckDB table.
    DuckDB reads directly from the Python list via its JSON path.
    """
    if not records:
        print(f"[{table_name}] no records – skipping.")
        return

    # Polars is used here because it handles mixed-type columns and nested
    # JSON gracefully before handing off to DuckDB.
    df = pl.from_dicts(records, infer_schema_length=500)
    con.register(table_name, df)
    # Materialise as a native DuckDB table so we can drop the Polars ref
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {table_name}")
    con.unregister(table_name)
    print(f"[{table_name}] {len(records):,} rows registered in DuckDB.")


# Use v2 data by default (swap raw1_v2 → raw1_v3 to use v3 ingestion)
records_to_duckdb(con, raw1_v2, "ds1")
records_to_duckdb(con, raw2_v2, "ds2")

## 3 · Inspect schemas

In [ ]:
def describe_table(con, table: str) -> pl.DataFrame:
    return con.execute(f"DESCRIBE {table}").pl()

print("=== ds1 schema ===")
ds1_schema = describe_table(con, "ds1")
print(ds1_schema)

print("\n=== ds2 schema ===")
ds2_schema = describe_table(con, "ds2")
print(ds2_schema)

In [ ]:
# Quick row-count + sample preview for both tables
for tbl in ("ds1", "ds2"):
    cnt = con.execute(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"\n── {tbl}  ({cnt:,} rows) ──")
    print(con.execute(f"SELECT * FROM {tbl} LIMIT 3").pl())

## 4 · Identify join candidates

The cell below uses Polars to find columns whose names overlap between the two tables – a quick heuristic for potential join keys.

In [ ]:
cols1 = set(ds1_schema["column_name"].to_list())
cols2 = set(ds2_schema["column_name"].to_list())

shared = sorted(cols1 & cols2)
only1  = sorted(cols1 - cols2)
only2  = sorted(cols2 - cols1)

print(f"Shared columns ({len(shared)}): {shared}")
print(f"Only in ds1   ({len(only1)}): {only1}")
print(f"Only in ds2   ({len(only2)}): {only2}")

# ── Common geo-column patterns to watch for ────────────────────────────
GEO_KEYWORDS = [
    "latitude", "longitude", "lat", "lon", "lng",
    "location", "geom", "geometry", "the_geom",
    "point", "borough", "zip", "zipcode", "nta",
    "bbl", "bin", "block", "lot", "tract", "census",
]

def find_geo_cols(cols, keywords=GEO_KEYWORDS):
    return [c for c in cols if any(kw in c.lower() for kw in keywords)]

geo1 = find_geo_cols(cols1)
geo2 = find_geo_cols(cols2)
print(f"\nGeo-like columns in ds1: {geo1}")
print(f"Geo-like columns in ds2: {geo2}")

## 5 · Merge strategies

Choose the strategy that matches your join keys:

| Strategy | When to use |
|----------|-------------|
| **5-A** Exact key join | Both tables share a common ID column |
| **5-B** Spatial join (point-in-polygon) | One table has points, other has polygons |
| **5-C** Spatial join (nearest neighbour) | Join each point to its closest feature |
| **5-D** Cross-join + filter | Exploratory – no obvious key yet |

### 5-A · Exact key join

Edit `JOIN_KEY` to match the shared column found in §4.

In [ ]:
# ── Configure before running ──────────────────────────────────────────────
JOIN_KEY = shared[0] if shared else None   # e.g. 'unique_key', 'bbl', 'sensor_id'
JOIN_TYPE = "INNER"                         # INNER | LEFT | RIGHT | FULL
# ─────────────────────────────────────────────────────────────────────────

if JOIN_KEY:
    sql = f"""
    SELECT
        a.*,
        -- Exclude the duplicated key column from ds2
        {', '.join(f'b.{c}' for c in sorted(cols2 - {JOIN_KEY}))}
    FROM ds1 a
    {JOIN_TYPE} JOIN ds2 b
        ON a."{JOIN_KEY}" = b."{JOIN_KEY}"
    """
    merged_exact = con.execute(sql).pl()
    print(f"Exact join on '{JOIN_KEY}': {merged_exact.shape[0]:,} rows × {merged_exact.shape[1]} cols")
    print(merged_exact.head(3))
else:
    print("No shared columns found – skip to a spatial join strategy.")

### 5-B · Spatial join – point-in-polygon

Assumes `ds1` has lat/lon sensor points and `ds2` has WKT polygon geometries (e.g. neighbourhood boundaries).

In [ ]:
# ── Configure before running ──────────────────────────────────────────────
# ds1 point columns
LAT_COL  = "latitude"    # float column in ds1
LON_COL  = "longitude"   # float column in ds1

# ds2 polygon column (WKT string  OR  a GeoJSON-like text column)
GEOM_COL = "the_geom"    # column in ds2
# ─────────────────────────────────────────────────────────────────────────

sql_pip = f"""
SELECT
    a.*,
    b.* EXCLUDE ({GEOM_COL})
FROM ds1 a
JOIN ds2 b
    ON ST_Within(
        ST_Point(CAST(a."{LON_COL}" AS DOUBLE), CAST(a."{LAT_COL}" AS DOUBLE)),
        ST_GeomFromText(b."{GEOM_COL}")
    )
"""

try:
    merged_pip = con.execute(sql_pip).pl()
    print(f"Point-in-polygon join: {merged_pip.shape[0]:,} rows × {merged_pip.shape[1]} cols")
    print(merged_pip.head(3))
except Exception as e:
    print(f"Spatial join failed – update LAT_COL / LON_COL / GEOM_COL above.\nError: {e}")

### 5-C · Spatial join – nearest neighbour (KNN)

For each point in `ds1`, find the closest row in `ds2` (great-circle distance in metres).

In [ ]:
# ── Configure before running ──────────────────────────────────────────────
# ds1 point columns
LAT1 = "latitude"
LON1 = "longitude"

# ds2 point columns (if ds2 also has lat/lon centroids)
LAT2 = "latitude"
LON2 = "longitude"

MAX_DIST_M = 500   # max match distance in metres (set None to disable)
# ─────────────────────────────────────────────────────────────────────────

dist_filter = f"AND dist_m <= {MAX_DIST_M}" if MAX_DIST_M else ""

sql_knn = f"""
WITH ranked AS (
    SELECT
        a.*,
        b.* EXCLUDE ("{LAT2}", "{LON2}"),
        ST_Distance(
            ST_Point(CAST(a."{LON1}" AS DOUBLE), CAST(a."{LAT1}" AS DOUBLE))::GEOGRAPHY,
            ST_Point(CAST(b."{LON2}" AS DOUBLE), CAST(b."{LAT2}" AS DOUBLE))::GEOGRAPHY
        ) AS dist_m,
        ROW_NUMBER() OVER (
            PARTITION BY a.rowid
            ORDER BY
                ST_Distance(
                    ST_Point(CAST(a."{LON1}" AS DOUBLE), CAST(a."{LAT1}" AS DOUBLE))::GEOGRAPHY,
                    ST_Point(CAST(b."{LON2}" AS DOUBLE), CAST(b."{LAT2}" AS DOUBLE))::GEOGRAPHY
                )
        ) AS rn
    FROM ds1 a
    CROSS JOIN ds2 b
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
  {dist_filter}
"""

try:
    merged_knn = con.execute(sql_knn).pl()
    print(f"Nearest-neighbour join (≤{MAX_DIST_M}m): {merged_knn.shape[0]:,} rows × {merged_knn.shape[1]} cols")
    print(merged_knn.head(3))
except Exception as e:
    print(f"KNN join failed – update LAT1/LON1/LAT2/LON2 above.\nError: {e}")

### 5-D · Cross-join + filter (exploratory)

When there is no obvious key yet – pull a small cross-product and inspect it.

In [ ]:
sql_xjoin = """
SELECT a.*, b.*
FROM   (SELECT * FROM ds1 LIMIT 200) a
CROSS JOIN (SELECT * FROM ds2 LIMIT 200) b
LIMIT 500
"""

cross = con.execute(sql_xjoin).pl()
print(f"Cross-join sample: {cross.shape[0]:,} rows × {cross.shape[1]} cols")
cross.head(3)

## 6 · Export merged result

Write the merged dataset to Parquet (efficient, schema-preserving).

In [ ]:
OUTPUT_PATH = "merged_output.parquet"

# Replace `merged_exact` with whichever merged DataFrame you want to export
try:
    result_df = merged_exact          # swap to merged_pip / merged_knn as needed
    result_df.write_parquet(OUTPUT_PATH)
    print(f"Saved {len(result_df):,} rows to {OUTPUT_PATH}")
except NameError:
    print("No merged DataFrame yet – run one of the join cells first.")

In [ ]:
# Alternatively, write directly from DuckDB to Parquet (no Python intermediary)
# con.execute(f"""
#     COPY (
#         SELECT a.*, b.* EXCLUDE ("{JOIN_KEY}")
#         FROM ds1 a INNER JOIN ds2 b ON a."{JOIN_KEY}" = b."{JOIN_KEY}"
#     ) TO '{OUTPUT_PATH}' (FORMAT PARQUET)
# """)